3.2

In [1]:
# =============================================================================
# 파이토치 3장: 신경망 구현
# =============================================================================
# =============================================================================
# 1. 3장 실습에 필요한 라이브러리 설치 및 import
# =============================================================================
import torch # Tensor을 생성하고 조작하는 라이브러리
import torch.nn as nn # 신경망 구축을 위한 고수준 API
import torch.optim as optim # 최적화 알고리즘(Optimizer)을 제공하여 모델 파라미터를 업데이트하는 패키지
import torch.nn.functional as F
import torchvision # 이미지 처리 및 컴퓨터 비전 작업을 쉽게 수행할 수 있도록 지원하는 라이브러리
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt # Matplotlib은 2D 그래프를 그릴때 가장 많이 사용되는 Python 라이브러리
import numpy as np
# sklearn 사이킷런(scikit-learn) 다양한 머신러닝 알고리즘을 구현한 파이썬 라이브러리
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns # Matplotlib을 기반으로 한 파이썬 데이터 시각화 라이브러리

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# 시드 고정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)

PyTorch version: 2.10.0+cpu
CUDA available: False


Pytorch 구성요소-딥러닝의 정석 with 파이토치 3장

In [2]:
# =============================================================================
# 2. 퍼셉트론을 이용한 AND 게이트 구현
# =============================================================================
print("\n" + "="*60)
print("2. 퍼셉트론을 이용한 AND 게이트 구현")
print("="*60)

# 단일 퍼셉트론을 이용한 AND 게이트 구현
def perceptron_and(x1, x2):
    w = torch.tensor([0.5, 0.5])  # 가중치
    b = torch.tensor(-0.7)        # 편향
    x = torch.tensor([x1, x2], dtype=torch.float32)
    output = torch.dot(w, x) + b  # 가중치 합 연산
    return 1 if output >= 0 else 0

# 테스트
print("AND 게이트 테스트:")
print(f"perceptron_and(0, 0) = {perceptron_and(0, 0)}")  # 0
print(f"perceptron_and(0, 1) = {perceptron_and(0, 1)}")  # 0
print(f"perceptron_and(1, 0) = {perceptron_and(1, 0)}")  # 0
print(f"perceptron_and(1, 1) = {perceptron_and(1, 1)}")  # 1


2. 퍼셉트론을 이용한 AND 게이트 구현
AND 게이트 테스트:
perceptron_and(0, 0) = 0
perceptron_and(0, 1) = 0
perceptron_and(1, 0) = 0
perceptron_and(1, 1) = 1


torch.dot(w, x): 두 개의 1차원 텐서(벡터) 간의 내적(Dot Product)을 계산하는 함수

In [3]:
import torch

w = torch.tensor([1.0, 2.0, 3.0])
x = torch.tensor([4.0, 5.0, 6.0])

# 내적 계산: (1*4) + (2*5) + (3*6) = 4 + 10 + 18 = 32
result = torch.dot(w, x)
print(result)  # 출력: tensor(32.)

tensor(32.)


one-line ternary expression

Python Style:  value_if_true if condition else value_if_false

*   Example: status = "adult" if age >= 18 else "minor"


3.3

In [4]:
# =============================================================================
# 3. 순전파 구현
# =============================================================================
print("\n" + "="*60)
print("3. 파이토치에서의 순전파 구현")
print("="*60)

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 3)  # 입력층(2) → 은닉층(3)
        self.fc2 = nn.Linear(3, 1)  # 은닉층(3) → 출력층(1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))      # 활성화 함수 적용
        print(f'x: {x}')
        x = torch.sigmoid(self.fc2(x))
        print(f'x: {x}')
        return x

#torch.manual_seed(42)
model = SimpleNN()
sample_input = torch.tensor([1.0, 2.0])
output = model(sample_input) # forward() 실행
print(f"SimpleNN 출력: {output}")


3. 파이토치에서의 순전파 구현
x: tensor([1.3702, 1.5487, 0.7538], grad_fn=<ReluBackward0>)
x: tensor([0.6693], grad_fn=<SigmoidBackward0>)
SimpleNN 출력: tensor([0.6693], grad_fn=<SigmoidBackward0>)


p81 - 88

3.4

In [5]:
# =============================================================================
# 4. 기본 신경망 클래스 설계
# =============================================================================
print("\n" + "="*60)
print("4. 기본 신경망 클래스 설계")
print("="*60)

class SimpleNN_Full(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN_Full, self).__init__() # to call the constructor of its parent class(superclass)
        self.fc1 = nn.Linear(input_size, hidden_size)   # 입력층 → 은닉층
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)  # 은닉층 → 출력층

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNN_Full(input_size=10, hidden_size=20, output_size=1)
print(f'model:\n {model}\n')

# 파라미터 업데이트 및 초기화
for param in model.parameters():
    print(f'param.shape:\n {param.shape}')
    #print(f'param:\n {param}')


4. 기본 신경망 클래스 설계
model:
 SimpleNN_Full(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=20, out_features=1, bias=True)
)

param.shape:
 torch.Size([20, 10])
param.shape:
 torch.Size([20])
param.shape:
 torch.Size([1, 20])
param.shape:
 torch.Size([1])


 torch.Size([20, 10]): input-hidden;  torch.Size([20]): bias;

 torch.Size([1, 20]): hidden-output;  torch.Size([1]): bias


'(케라스에서 제공하는) activation 함수의 종류 2' 파일 참조


In [6]:
# =============================================================================
# 5. 데이터 로더 설정
# =============================================================================
print("\n" + "="*60)
print("5. 데이터 로더 설정")
print("="*60)

X_train = torch.randn(100, 10)
y_train = torch.randn(100, 1)
dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


5. 데이터 로더 설정


torch.rand(100, 10): 0과 1 사이의 균등분포(Uniform Distribution) 난수 생성.
*   100행 10열 (100개의 샘플sample, 각 샘플은 10개의 특징feature/차원을 가짐)

torch.randn(100, 10): 표준정규분포(Normal Distribution) 난수 생성.

torch.randint(0, 2, (100, 10)): 0과 1 사이의 정수 난수 생성.

*  low (0): 생성될 난수의 최소값 (포함).
*  high (2): 생성될 난수의 최대값 (미포함).

즉, 0 이상 2 미만인 정수, 0 또는 1이 생성됩니다.



dataset = TensorDataset(X_train, y_train): to bundle multiple tensors into a single dataset object

batch_size=32: grouped into mini-batches of 32 samples each.

shuffle=True: 매 에포크 시작 시 전체 데이터셋을 다시 섞습니다. 이는 모델이 데이터 순서를 학습하는 것을 방지하고 오버헤드를 줄이는 데 도움이 됩니다.

In [7]:
# =============================================================================
# 6. 학습 및 검증 과정 구현
# =============================================================================
print("\n" + "="*60)
print("6. 학습 및 검증 과정 구현")
print("="*60)

criterion = nn.MSELoss() # 평균 제곱 오차(Mean Squared Error) 손실 함수
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    for batch in dataloader:
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


6. 학습 및 검증 과정 구현
Epoch 1, Loss: 0.9306035041809082
Epoch 2, Loss: 1.979332447052002
Epoch 3, Loss: 0.524667501449585
Epoch 4, Loss: 0.9314338564872742
Epoch 5, Loss: 2.472566843032837
Epoch 6, Loss: 1.977538824081421
Epoch 7, Loss: 2.6563892364501953
Epoch 8, Loss: 1.100563645362854
Epoch 9, Loss: 1.5428667068481445
Epoch 10, Loss: 0.08924790471792221


nn.MSELoss(): 평균 제곱 오차(Mean Squared Error) 손실 함수

optimizer = optim.Adam(model.parameters(), lr=0.001): initializes the Adam (Adaptive Moment Estimation) optimizer

model.parameters(): 이 속성은 옵티마이저에게 학습 중에 업데이트해야 하는 모델 내의 모든 학습 가능한 가중치와 편향을 담은 반복 가능한 객체를 제공

lr=0.001: Sets the learning rate (step size)


optimizer.zero_grad(): gradients 값들을 backward를 해줄때 계속 더해주기 때문에 항상 backpropagation을 하기전에 gradients를 zero로 만들어주고 시작을 해야함-딥러닝의 정석 with 파이토치 3장

이렇게 gradients을 더해주는 방식은 RNN을 학습시킬때 매우 편리한 방식

loss.backward(): 자동 미분 엔진인 autograd를 사용하여 계산 그래프의 끝단(loss)에서부터 역방향으로 각 파라미터(requires_grad=True인 텐서)에 대한 기울기(Gradient)를 계산하는 핵심 함수

optimizer.step(): loss.backward()를 통해 계산된 그래디언트(gradient)(변화도)를 사용하여 모델의 매개변수(가중치)를 업데이트하는 핵심 함수

In [8]:
# =============================================================================
# 7. 체크 포인트 저장과 불러오기
# =============================================================================
print("\n" + "="*60)
print("7. 체크 포인트 저장과 불러오기")
print("="*60)

torch.save(model.state_dict(), 'model.pth')
model = SimpleNN_Full(input_size=10, hidden_size=20, output_size=1)
model.load_state_dict(torch.load('model.pth'))


7. 체크 포인트 저장과 불러오기


<All keys matched successfully>

torch.save(model.state_dict(), 'model.pth'): 학습된 모델의 가중치(weights)와 편향(bias) 정보인 state_dict를 'model.pth' 파일로 저장. 모델 구조 전체가 아닌 파라미터만 저장-파일 용량이 가볍고, 나중에 유연하게 모델을 불러와 활용할 수 있음

model.load_state_dict(torch.load('model.pth')): 학습된 모델의 가중치(state_dict)를 파일(model.pth)로부터 불러와 모델 인스턴스에 적용하는 방식. 저장된 파라미터(weights, biases)를 모델 구조에 매핑하여 추론 또는 추가 학습에 활용

.pth 파일: 체크포인트와 가중치 저장에 사용되는 파일

.pth는 체크포인트 또는 학습된 가중치를 저장할 때 주로 사용. PyTorch 커뮤니티에서는 .pth 확장자를 통해 학습 중간 결과나 최종 가중치를 저장하는 것이 관례.

모델을 배포하거나, 다른 환경에서 추론만 수행하려는 경우: .pt 확장자를 사용하는 것이 적합. 모델 전체 구조와 가중치를 함께 저장하면 배포 시 유리하다.

In [9]:
# =============================================================================
# 8. 손실값 NaN 처리
# =============================================================================
print("\n" + "="*60)
print("8. 학습 중 디버깅 - NaN 처리")
print("="*60)

# 손실값이 NaN이 되는 문제 해결
def check_nan_loss(loss, optimizer):
    if torch.isnan(loss).any():
        print("Loss contains NaN values! Reducing learning rate...")
        for param_group in optimizer.param_groups:
            param_group['lr'] *= 0.1
        return True
    return False

# 예시 사용법
sample_loss = torch.tensor(float('nan'))
nan_detected = check_nan_loss(sample_loss, optimizer)
print(f"NaN 감지됨: {nan_detected}")


8. 학습 중 디버깅 - NaN 처리
Loss contains NaN values! Reducing learning rate...
NaN 감지됨: True


 torch.isnan(loss).any(): 손실값(loss) 텐서 내에 단 하나라도 NaN(Not a Number, 숫자가 아님) 값이 존재하는지 확인하는 코드

*   torch.isnan(loss): 텐서의 각 요소가 NaN인지 여부를 Boolean(True/False) 텐서로 반환
*   .any(): Boolean 텐서 중 하나라도 True이면 True를 반환하고, 모두 False이면 False를 반환

NaN(Not A Number, /næn/): 연산 과정에서 잘못된 입력을 받았음을 나타내는 기호

print("Loss contains NaN values! Reducing learning rate..."):

손실 함수(Loss)가 발산하여 계산 불가능한 값((NaN)이 되었을 때 발생.
주로 그래디언트가 너무 커져서 파라미터가 급격히 변하거나, 데이터에 문제가 있을 때 나타납니다.
이 문제를 해결하기 위한 단계별 대처 방안은 다음과 같습니다.
1. 즉시 적용 가능한 해결책 (가장 일반적)Learning Rate (학습률) 낮추기: 현재 설정된 LR이 너무 클 수 있습니다. 기존보다 1/10 또는 1/5 수준으로 낮춰서 다시 시도합니다.
2. Batch Size (배치 사이즈) 키우기: 배치 사이즈를 키우는 것은 러닝 레이트를 줄이는 것과 유사한 안정화 효과를 냅니다.그래디언트의 "노이즈" 감소 (Signal-to-Noise Ratio 향상)

*    큰 배치(Large Batch): 더 많은 데이터를 평균 내어 기울기를 계산하므로, 데이터의 분포를 더 정확하게 반영하는 정확한 그래디언트(True Gradient)에 가까워집니다.
*   안정화 효과: 노이즈가 적기 때문에 학습이 덜 불안정하게 진행되며, 급격한 손실(Loss) 변화없이 안정적으로 수렴 방향을 찾습니다.

3. Gradient Clipping (그래디언트 클리핑) 적용: 그래디언트의 최대치를 제한하여 폭발(Exploding Gradient)을 막습니다.


optimizer.param_groups: 최적화하려는 매개변수(Parameters)들의 그룹과 그에 따른 하이퍼파라미터 설정을 담고 있는 리스트(list)
각 원소는 딕셔너리(dict) 형태이며, 'params'(학습할 텐서 리스트), 'lr'(학습률), 'weight_decay' 등의 키값을 가집니다.

Regularization Strategies, weight_decay-딥러닝의 정석 with 파이토치 3장

In [10]:
# =============================================================================
# 9. 데이터 증강 기법
# =============================================================================
print("\n" + "="*60)
print("9. 데이터 증강 기법")
print("="*60)

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

print("데이터 증강 변환이 설정되었습니다.")


9. 데이터 증강 기법
데이터 증강 변환이 설정되었습니다.


Data Augmentation-딥러닝의 정석 with 파이토치 3장

3.4 실습예제 p93

In [11]:
# =============================================================================
# 10. nn.Module 클래스를 이용한 신경망 구현
# =============================================================================
print("\n" + "="*60)
print("10. nn.Module 클래스를 이용한 신경망 구현")
print("="*60)

# 데이터셋 생성
X_train = torch.randn(100, 10)
y_train = torch.randn(100, 1)
dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 신경망 모델 정의
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 모델 초기화
model = SimpleNN(input_size=10, hidden_size=20, output_size=1)
print(model)

# 손실 함수 및 옵티마이저 설정
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 루프 구현
for epoch in range(10):
    for batch in dataloader:
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

# 모델 저장 및 불러오기
torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))

# 학습 중 NaN값 확인 및 학습률 조정
if torch.isnan(loss).any():
  print("Loss contains NaN values! Reducing learning rate...")
  for param_group in optimizer.param_groups:
    param_group['lr'] *= 0.1

# 데이터 증강 기법 적용(이미지 데이터의 경우)
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])


10. nn.Module 클래스를 이용한 신경망 구현
SimpleNN(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=20, out_features=1, bias=True)
)
Epoch 1, Loss: 0.35286468267440796
Epoch 2, Loss: 2.3873112201690674
Epoch 3, Loss: 0.16931483149528503
Epoch 4, Loss: 0.511035144329071
Epoch 5, Loss: 1.5744847059249878
Epoch 6, Loss: 0.17984656989574432
Epoch 7, Loss: 0.18049563467502594
Epoch 8, Loss: 2.355794668197632
Epoch 9, Loss: 1.9445692300796509
Epoch 10, Loss: 1.2623677253723145


3.5 MNIST

In [12]:
# =============================================================================
# 파이토치 3장: 신경망 구현
# =============================================================================
# =============================================================================
# 1. 3장 실습에 필요한 라이브러리 설치 및 import
# =============================================================================
import torch # Tensor을 생성하고 조작하는 라이브러리
import torch.nn as nn # 신경망 구축을 위한 고수준 API
import torch.optim as optim # 최적화 알고리즘(Optimizer)을 제공하여 모델 파라미터를 업데이트하는 패키지
import torch.nn.functional as F
import torchvision # 이미지 처리 및 컴퓨터 비전 작업을 쉽게 수행할 수 있도록 지원하는 라이브러리
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt # Matplotlib은 2D 그래프를 그릴때 가장 많이 사용되는 Python 라이브러리
import numpy as np
# sklearn 사이킷런(scikit-learn) 다양한 머신러닝 알고리즘을 구현한 파이썬 라이브러리
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns # Matplotlib을 기반으로 한 파이썬 데이터 시각화 라이브러리

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# 시드 고정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)

PyTorch version: 2.10.0+cpu
CUDA available: False


In [13]:
# =============================================================================
# 11. MNIST 데이터 전처리
# =============================================================================
print("\n" + "="*60)
print("11. MNIST 데이터 전처리")
print("="*60)

# 데이터 변환 설정
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # 평균과 표준 편차를 사용한 정규화
])

# MNIST 데이터셋 불러오기
train_dataset = torchvision.datasets.MNIST(root='./data', train=True,
                                          transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False,
                                         transform=transform, download=True)


11. MNIST 데이터 전처리


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 473kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.26MB/s]


In [14]:
# =============================================================================
# 12.배치 처리 및 데이터 로더 사용
# =============================================================================
print("\n" + "="*60)
print("12.배치 처리 및 데이터 로더 사용")
print("="*60)

# 배치 처리 및 데이터 로더 사용
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

print(f"훈련 데이터 크기: {len(train_dataset)}")
print(f"테스트 데이터 크기: {len(test_dataset)}")

#print(f"훈련 데이터: {train_dataset[0]}")


12.배치 처리 및 데이터 로더 사용
훈련 데이터 크기: 60000
테스트 데이터 크기: 10000


In [15]:
# =============================================================================
# 13. MLP 기반 MNIST 분류기 구현
# =============================================================================
print("\n" + "="*60)
print("13. MLP 기반 MNIST 분류기 구현")
print("="*60)

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)  # 입력층 → 은닉층
        self.fc2 = nn.Linear(128, 64)     # 은닉층 → 은닉층
        self.fc3 = nn.Linear(64, 10)      # 은닉층 → 출력층

    def forward(self, x):
        x = x.view(-1, 28*28)  # 이미지를 1D 벡터로 변환
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


13. MLP 기반 MNIST 분류기 구현


In [16]:
# =============================================================================
# 14. 하이퍼파라미터 설정 및 실험
# =============================================================================
print("\n" + "="*60)
print("14. 하이퍼파라미터 설정 및 실험")
print("="*60)

# 하이퍼파라미터 설정 및 실험
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


14. 하이퍼파라미터 설정 및 실험


In [ ]:
# =============================================================================
# 15. 학습 진행 및 실시간 모니터
# =============================================================================
print("\n" + "="*60)
print("15. 학습 진행 및 실시간 모니터")
print("="*60)

for epoch in range(10):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


15. 학습 진행 및 실시간 모니터
Epoch 1, Loss: 0.07826259732246399
Epoch 2, Loss: 0.010672305710613728
Epoch 3, Loss: 0.0732647255063057
Epoch 4, Loss: 0.07901225239038467


실습 전체 코드

In [ ]:
# =============================================================================
# 16. MNIST 분류기 실습 전체 코드
# =============================================================================

print("\n" + "="*60)
print("16. MNIST 분류기 실습 전체 통합")
print("="*60)

# 데이터 변환 설정
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# MNIST 데이터셋 불러오기
train_dataset = torchvision.datasets.MNIST(root='./data', train=True,
                                          transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False,
                                         transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

# MLP 모델 정의
class MLP_Final(nn.Module):
    def __init__(self):
        super(MLP_Final, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP_Final().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 최종 학습
print("최종 모델 학습 중...")
for epoch in range(3):  # 빠른 실행을 위해 3 에포크
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 300 == 0:
            print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')